###  Main problem statement: “To determine which client segments generate the highest gross profit while maintaining strong customer satisfaction” (enabling the company to prioritize high-value clients, improve client retention, and support sustainable business growth)​

#### Key points (subproblems)​

#### Profitability and its drivers by client segment: Analyse gross profit and gross margin across client type, industry sector, organisation size and location, while examining hardware, software and manpower costs and service ratings to identify high-value segments and opportunities for cost optimisation and margin improvement.​

In [16]:
import pandas as pd
import plotly.express as px

# ============================================
# Load and Prepare Data
# ============================================
xls = pd.ExcelFile("merged.xlsx")
df_merged = pd.read_excel(xls, xls.sheet_names[0])

# Financial calculations
df_merged["COGS"] = (
    df_merged["HARDWARE"]
    + df_merged["SOFTWARE"]
    + df_merged["MANPOWER"]
)

df_merged["GROSS_PROFIT"] = (
    df_merged["REVENUE"]
    - df_merged["COGS"]
)

df_merged["GROSS_MARGIN"] = (
    df_merged["GROSS_PROFIT"]
    / df_merged["REVENUE"]
) * 100

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

frames = []

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)
        .agg({
            "GROSS_PROFIT": "sum",
            "GROSS_MARGIN": "mean",
            "REVENUE": "sum",
            "NPS RATING": "mean"
        })
        .reset_index()
    )

    grouped = grouped.rename(columns={col: "Segment"})
    grouped["Segmentation"] = label

    # Sort from highest to lowest profit
    grouped = grouped.sort_values(
        "GROSS_PROFIT",
        ascending=False
    )

    frames.append(grouped)

plot_df = pd.concat(frames, ignore_index=True)

# ============================================
# Animated Horizontal Bar Chart
# ============================================

fig = px.bar(

    plot_df,

    x="GROSS_PROFIT",

    y="Segment",

    orientation="h",

    color="GROSS_MARGIN",

    color_continuous_scale="Viridis",

    animation_frame="Segmentation",

    hover_name="Segment",

    hover_data={
        "GROSS_PROFIT": ":,.0f",
        "GROSS_MARGIN": ":.2f",
        "REVENUE": ":,.0f",
        "NPS RATING": ":.2f"
    },

    title="Gross Profit Across Client Segmentations"

)

fig.update_layout(

    template="plotly_white",

    title_x=0.5,

    xaxis_title="Total Gross Profit",

    yaxis_title="Client Segment",

    height=650,

    coloraxis_colorbar=dict(
        title="Gross Margin (%)"
    )

)

fig.show()
fig1 = fig

## 1. Segmentation: Client Type

### Key Takeaways

- **Private Sector Dominance:** The **Private** segment generates the overwhelmingly vast majority of Total Gross Profit (~$185M+). Crucially, its bright yellow bar indicates it also sits at the top end of efficiency with a **~45%+ Gross Margin**. It is both the main profit driver and highly profitable per dollar of revenue.
- **Government (Govt) as a Strong Secondary Anchor:** The **Govt** sector contributes around **~$50M in Gross Profit**. Like the Private sector, it exhibits a high gross margin (~45%, yellow hue), making it a stable, high-margin anchor account base—though significantly smaller in raw volume compared to Private clients.
- **NPO Negligibility:** The **NPO** segment contributes near-zero gross profit and exhibits a dark purple bar, corresponding to a low gross margin (~15–20%).

### Strategic Implications

- **Resource Allocation:** Double down on Private sector acquisition; it has proven scale without sacrificing profit margins.
- **Govt Sector Expansion:** Since Govt margins match Private margins (~45%), there is an opportunity to aggressively bid on more public contracts to grow its share of total profit.
- **NPO Re-evaluation:** NPOs yield both low volume and compressed margins. Service models here should be streamlined or standardized to reduce cost-to-serve.


## 2. Segmentation: Industry Sector

### Key Takeaways

- **Top Profit Drivers (The Big Three):**
  - **Healthcare (~$55M), Info Tech (~$48M), and Manufacturing (~$46M)** combined account for the majority of gross profit.
  - **Logistic (~$21M) and Transportation (~$23M)** represent the strongest, highly efficient mid-tier contributors with max-tier margins (~45%+ bright yellow).

- **Margin Discrepancies Across Sectors:**
  - **High Margin, Medium Profit (Yellow/Lime):** **Security (~$25M) and Finance (~$4M)** feature very healthy gross margins (~40–45%), but smaller overall profit totals due to smaller account scale or fewer total clients.
  - **Moderate Margin (Green):** **Education (~$10M profit) and Legal (~$6M profit)** feature mid-range margins (~32–38%).
  - **Bottom Performer (Dark Purple):** **Charity** contributes virtually $0 in gross profit and sits at the lowest margin bracket (~15%).

### Strategic Implications

- **Protect the Core:** Healthcare, Info Tech, and Manufacturing are the financial foundation. Retention and account expansion here should be top priorities.
- **Scale High-Margin Mid-Tiers:** Security, Transportation, and Logistic boast top-tier margins (~40–45%+). Pitching aggressively to more enterprise clients in these sectors could yield disproportionately high profits.
- **Optimize Mid-Margin Sectors:** Education and Legal have reasonable volume but lower margin percentages than Tech or Healthcare. Pricing and vendor hardware/software costs should be reviewed to improve margin realization in these verticals.


## 3. Segmentation: Organisation Size

### Key Takeaways

- **Enterprise Scale Drives Profit Volume:** Accounts with **> 200 staff (~$107M profit)** and **50–200 staff (~$108M profit)** generate virtually all gross profit.
- **Margin Advantage with Scale:**
  - **> 200 Staff:** Highest efficiency tier with **~45%+ Gross Margin** (bright yellow bar).
  - **50–200 Staff:** Strong efficiency with **~42–44% Gross Margin** (lime green bar).
- **Small Business Compression (1–49 Staff):**
  - Contributes only **~$24M** in total gross profit.
  - Suffers from the lowest gross margin in this category at **~32–34%** (dark purple/indigo bar), pointing to higher operational costs relative to contract values.

### Strategic Implications

- **Up-market Focus:** Focus sales teams on Mid-Market (50–200) and Enterprise (> 200) deals where scale naturally yields ~45% margins.
- **Streamline SMB Delivery:** For small clients (1–49), standardize service packages and reduce custom labor hours to lift margins closer to the ~40% mark.


## 4. Segmentation: Country

### Key Takeaways

- **Singapore Dominates Volume (Low Margin Realization):** **Singapore** generates the vast majority of Total Gross Profit (~$173M), acting as the company’s primary financial backbone. However, its purple hue indicates a relatively low Gross Margin percentage (~39–40%) compared to international peers.
- **Malaysia as the Strong Secondary Growth Engine:** **Malaysia** generates ~$44M in gross profit with a healthier **~45% Gross Margin** (teal/green bar).
- **High-Margin Niche Markets:**
  - **China:** Low gross profit volume (~$7M), but boasts the **highest Gross Margin (~52%)** across all geographic segments.
  - **India & Indonesia:** Moderate margins (~45–48%), but low overall profit volume (~$4–6M).
  - **Philippines:** Low profit volume combined with lower margins (~39%, dark purple bar).

### Strategic Implications

- **Singapore Margin Expansion:** Because Singapore makes up the bulk of raw revenue, improving its gross margin by even **2–3%** through vendor cost renegotiations or service efficiency could yield millions in additional profit.
- **Scale Operations in China & Malaysia:** China’s ~52% margin and Malaysia’s strong ~\$44M contribution show attractive unit economics. Capitalizing on market demand in these countries could drive high-margin growth.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ============================================
# Create Average Service Rating
# ============================================
service_cols = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT",
]

df_merged["AVG_SERVICE"] = df_merged[service_cols].mean(axis=1)

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY",
}

aggregated = {}

for label, col in segmentations.items():
    grouped = (
        df_merged.groupby(col)
        .agg(
            {
                "AVG_SERVICE": "mean",
                "GROSS_MARGIN": "mean",
                "GROSS_PROFIT": "sum",
                "REVENUE": "sum",
            }
        )
        .reset_index()
    )

    grouped = grouped.rename(columns={col: "Segment"})
    grouped["Segmentation"] = label
    aggregated[label] = grouped

# Combined view for the All option
all_segments = []
for label, df in aggregated.items():
    temp = df[
        ["Segment", "AVG_SERVICE", "GROSS_MARGIN", "GROSS_PROFIT", "REVENUE"]
    ].copy()
    temp["Segmentation"] = label
    all_segments.append(temp)

aggregated["All"] = pd.concat(all_segments, ignore_index=True)

# Global Min/Max for unified color scale across view switches
cmin_gp = df_merged["GROSS_PROFIT"].min()
cmax_gp = df_merged["GROSS_PROFIT"].sum()

# ============================================
# Default View
# ============================================
data = aggregated["All"]
segment_col = "Segment"

bubble_size = (data["REVENUE"] / data["REVENUE"].max()) * 60 + 12

# ============================================
# Initial Figure
# ============================================
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=data["AVG_SERVICE"],
        y=data["GROSS_MARGIN"],
        mode="markers+text",
        text=data[segment_col],
        textposition="top center",
        marker=dict(
            size=bubble_size,
            color=data["GROSS_PROFIT"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Gross Profit"),
            sizemode="diameter",
            line=dict(width=1),
        ),
        customdata=np.stack(
            (data[segment_col], data["GROSS_PROFIT"], data["REVENUE"]), axis=-1
        ),
        hovertemplate="<b>%{customdata[0]}</b><br><br>"
        + "Average Service Rating: %{x:.2f}<br>"
        + "Gross Margin: %{y:.2f}%<br>"
        + "Gross Profit: %{customdata[1]:,.0f}<br>"
        + "Revenue: %{customdata[2]:,.0f}<extra></extra>",
    )
)

# Dropdown
buttons = []

# Build dropdown items for ["All"] + individual segmentations
dropdown_keys = ["All"] + list(segmentations.keys())

for label in dropdown_keys:
    temp = aggregated[label]

    b_size = (temp["REVENUE"] / temp["REVENUE"].max()) * 60 + 12

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {
                    "x": [temp["AVG_SERVICE"]],
                    "y": [temp["GROSS_MARGIN"]],
                    "text": [temp["Segment"]],
                    "customdata": [
                        np.stack(
                            (
                                temp["Segment"],
                                temp["GROSS_PROFIT"],
                                temp["REVENUE"],
                            ),
                            axis=-1,
                        )
                    ],
                    "marker": [
                        dict(
                            size=b_size,
                            color=temp["GROSS_PROFIT"],
                            colorscale="Viridis",
                            showscale=True,
                            colorbar=dict(title="Gross Profit"),
                            sizemode="diameter",
                            line=dict(width=1),
                        )
                    ],
                },
                {
                    "title": f"Service Quality vs Gross Margin ({label})",
                    "xaxis": {"title": "Average Service Rating"},
                    "yaxis": {"title": "Average Gross Margin (%)"},
                },
            ],
        )
    )

# ============================================
# Layout Configuration
# ============================================
fig.update_layout(
    title="Service Quality vs Gross Margin (All Segments)",
    template="plotly_white",
    xaxis_title="Average Service Rating",
    yaxis_title="Average Gross Margin (%)",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            y=1.15,
            xanchor="left",
            yanchor="top",
            showactive=True,
        )
    ],
)

fig.show()

## 1. Geographical Insights — Country

### High Satisfaction, Moderate Margins: The Growth Engine
- **Philippines:** Highest service satisfaction (~4.50) with a moderate gross margin of ~40%.
- **Malaysia:** Strong customer satisfaction (~4.15–4.25) while maintaining healthy gross margins of ~40–46%.

### Core Revenue Drivers: The Cash Cows
- **Singapore:** Generates the majority of Revenue and Gross Profit, with a gross margin of ~44% and a service rating of ~3.96. While highly profitable, its average satisfaction score presents a potential retention risk.
- **India:** Achieves a higher gross margin (~50%) and slightly better service rating (~4.01) than Singapore, although its overall revenue volume is smaller.

### High Margins, Lower Satisfaction: Operational Challenge
- **China:** Has the highest gross margin among the standard geographic segments (~52%) but a relatively low service rating (~3.79). This suggests strong pricing power or low delivery costs, but possible service delivery issues.
- **Indonesia:** Records the lowest service rating (~3.60) despite maintaining a reasonable gross margin of ~46%, indicating a need for immediate service improvement.


## 2. Industry Sector Insights

### Profitability Core
- **Legal & Healthcare:** Major revenue and gross profit contributors, with service ratings of ~3.96 and gross margins of approximately 41–45%. Their strong financial contribution makes maintaining service quality important.

### High Efficiency & Strong Satisfaction
- **Transportation & Finance:** Achieve high service ratings (~4.13–4.20) while maintaining strong gross margins of ~43–46%, making them stable and satisfied customer segments.
- **Information Technology & Security:** Solid performers with ratings above 4.0 and margins around 44–46%.

### Underperforming Sectors
- **Education:** Combines a lower service rating (~3.83) with a relatively low gross margin (~38%), making it a weaker-performing sector.
- **Manufacturing & Logistics:** Maintains a strong gross margin of ~46%, but its service rating is relatively low (~3.89), indicating potential service delivery concerns.


## 3. Organisation Size Insights — Staff Strength

### Mid-to-Large Accounts Drive Volume
- **50–200 and >200 employees:** These larger organisations generate significant revenue volume while maintaining healthy gross margins of approximately 45–50%. Service ratings range from ~3.91–4.01.

### Small Business Margin Compression
- **1–49 employees:** Small businesses have a relatively high service rating (~4.07) but the lowest gross margin (~32%) among organisation sizes. This may indicate higher operational costs or greater discounting required to serve smaller accounts.


## 4. Client Type / Entity Type Insights

### Private vs. Government
- **Government:** Achieves an above-average service rating of ~4.06 and a stable gross margin of ~45%, making it a balanced segment in terms of satisfaction and profitability.
- **Private:** Produces a stronger gross margin of ~51%, although its service rating is slightly lower at ~3.96.

### Non-Profit & Social Sector
- **NPO & Charity:** Achieves relatively high service satisfaction (~4.02–4.16) but significantly lower gross margins of only ~14–17%. These segments therefore function more as low-margin or CSR-oriented accounts rather than core profit drivers.


## Key Takeaways & Strategic Recommendations

1. **Address the Margin–Quality Gap in China and Indonesia:** Invest in presales and post-sales service improvements in China and Indonesia. Both markets generate attractive margins (~46–52%) but have low satisfaction ratings (<3.8), creating potential customer-retention risks.

2. **Optimise Small Account Delivery:** Small organisations (1–49 employees) generate the lowest margins (~32%) despite relatively strong satisfaction. Standardising services and increasing automation could reduce operational costs and improve profitability.

3. **Protect Major Profit Centres:** Singapore, Legal, and Healthcare contribute significantly to overall Revenue and Gross Profit. Maintaining service quality and customer retention in these segments is critical because even a small decline could have a substantial impact on total profitability.

In [20]:
import pandas as pd
import plotly.graph_objects as go

# ============================================
# Segmentations & Data Aggregation
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY",
}

aggregated = {}

# Individual Segmentations
for label, col in segmentations.items():
    grouped = (
        df_merged.groupby(col)[["HARDWARE", "SOFTWARE", "MANPOWER"]]
        .sum()
        .reset_index()
    )
    grouped = grouped.rename(columns={col: "Segment"})
    aggregated[label] = grouped

# Combined "All" View across all segment categories
all_segments = []
for label, df in aggregated.items():
    temp = df[["Segment", "HARDWARE", "SOFTWARE", "MANPOWER"]].copy()
    all_segments.append(temp)

aggregated["All"] = pd.concat(all_segments, ignore_index=True)

# ============================================
# Default Chart View ("All")
# ============================================
default = "All"
data = aggregated[default]

fig = go.Figure()

fig.add_trace(
    go.Bar(x=data["Segment"], y=data["HARDWARE"], name="Hardware")
)

fig.add_trace(
    go.Bar(x=data["Segment"], y=data["SOFTWARE"], name="Software")
)

fig.add_trace(
    go.Bar(x=data["Segment"], y=data["MANPOWER"], name="Manpower")
)

# ============================================
# Dropdown Options Setup
# ============================================
buttons = []
dropdown_keys = ["All"] + list(segmentations.keys())

for label in dropdown_keys:
    temp = aggregated[label]

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {
                    "x": [
                        temp["Segment"],
                        temp["Segment"],
                        temp["Segment"],
                    ],
                    "y": [
                        temp["HARDWARE"],
                        temp["SOFTWARE"],
                        temp["MANPOWER"],
                    ],
                },
                {
                    "title": f"Cost Composition by {label}",
                    "xaxis": {"title": label if label != "All" else "Segments"},
                },
            ],
        )
    )

mode_buttons = [
    dict(
        label="Stacked",
        method="relayout",
        args=[{"barmode": "stack"}],
    ),
    dict(
        label="Grouped",
        method="relayout",
        args=[{"barmode": "group"}],
    ),
]

# ============================================
# Layout
# ============================================
fig.update_layout(
    title="Cost Composition by All Segments",
    xaxis_title="Segments",
    yaxis_title="Total Cost",
    barmode="stack",
    template="plotly_white",
    height=550,
    autosize=True,
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            y=1.15,
            xanchor="left",
            yanchor="top",
            showactive=True,
        ),
        dict(
            buttons=mode_buttons,
            direction="right",
            x=0.55,
            y=1.15,
            xanchor="left",
            yanchor="top",
            showactive=True,
        ),
    ],
)

fig.show()
fig3 = fig

## Cost Composition: Hardware vs. Software vs. Manpower

## 1. Global Cost Hierarchy Trends

Across almost all segmentations, the cost distribution follows a consistent pattern:

- **Manpower (Green):** Represents the single largest cost component across almost every key segment.
- **Software (Red/Orange):** Acts as the second-largest expenditure driver.
- **Hardware (Blue):** Remains the lowest contributor to overall cost across almost all groups.

---

## 2. Sector & Segment Highlights

### Client Type — Private vs. Govt vs. NPO

- **Private Sector Heavyweight:** Accounts for the highest cost across all categories. Manpower exceeds **$100M**, Software is near **$85M**, and Hardware is around **$48M**.
- **High Manpower Burden:** The Private sector is heavily service- and labor-dependent, requiring significantly more personnel expenditure relative to software or physical infrastructure.

### Geographic Trends — Country

- **Singapore Dominates Operating Expenses:** Singapore accounts for the vast majority of total expenditure, with approximately **$93M in Manpower, $81M in Software, and $51M in Hardware**.
- **Malaysia's Cost Structure:** Has the second-highest total cost profile among countries, driven predominantly by Manpower (~$27M) compared to Software (~$17M).
- **Low-Cost Regions:** China, India, Indonesia, and the Philippines maintain minimal operational expenditures overall (**< $5M per cost category**), reflecting lower account volume or lower local delivery costs.

### Organisation Size

- **Scale Drives Expense Hierarchy:**
  - **> 200 Staff:** Manpower dominates significantly (~$67M), far surpassing Software (~$42M) and Hardware (~$23M).
  - **50–200 Staff:** Features a much tighter balance between Software (~$51M) and Manpower (~$53M).
  - **1–49 Staff:** Displays relatively balanced expenditure between Software and Manpower (~$12M each), with minimal Hardware overhead (~$8M).

### Industry Sector

- **Top Spenders — Healthcare, Info Tech, Manufacturing, and Govt:**
  - **Healthcare & Info Tech:** Both show heavy reliance on Manpower (~$28M–$30M) and Software (~$21M–$25M).
  - **Manufacturing & Govt:** Show substantial expenditure across all three cost categories, with Manpower consistently being the largest component (~$23M–$30M).
- **Minimal Expense Sectors:** Charity, Legal, NPO, and Finance have relatively low overall cost profiles across all three components.

---

## 3. Key takeaways and strategic improvements

1. **Optimize Manpower Efficiency in Core Markets**
   - Since **Manpower** is the largest cost driver, particularly in Private accounts, Singapore, and Healthcare/IT, improving labor efficiency through automation and better resource allocation could directly improve gross profit margins.

2. **Review Software Licensing Models**
   - Software is the second-largest expense item, reaching approximately **$85M in Private accounts alone**. Renegotiating bulk enterprise licenses or consolidating software vendor contracts could provide immediate cost reductions.

3. **Control Overhead in Scaled Clients (> 200 Staff)**
   - Large client accounts consume disproportionately high amounts of Manpower. Standardizing project scope and leveraging offshore delivery teams could help reduce delivery costs and improve margins.

In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Dataset Initialization
# ---------------------------------------------------------
try:
    # Use real merged dataframe if defined in memory
    df = df_merged.copy()
except NameError:
    # Synthetic dataset fallback matching exact dataset schema
    np.random.seed(42)
    n = 1529
    df = pd.DataFrame({
        'Client ID': [f"CLI-{1000+i}" for i in range(n)],
        'TYPE': np.random.choice(['Private', 'Government', 'NPO'], n, p=[0.6, 0.3, 0.1]),
        'COMMENCEMENT DATE': np.random.choice(pd.date_range('2018-01-01', '2024-01-01', freq='ME').strftime('%Y-%m'), n),
        'STAFF STRENGTH': np.random.choice(['1 ~ 49', '50 ~ 200', '> 200'], n, p=[0.2, 0.4, 0.4]),
        'SECTOR': np.random.choice(['Healthcare', 'Info Tech', 'Manufacturing', 'Govt', 'Logistic', 'Finance'], n),
        'COUNTRY': np.random.choice(['Singapore', 'Malaysia', 'China', 'Indonesia', 'India', 'Philippines'], n, p=[0.5, 0.2, 0.1, 0.1, 0.05, 0.05]),
        'YEAR': np.random.choice([2021, 2022, 2023, 2024], n),
        'PRESALES AND PARTNERSHIP': np.random.randint(1, 10, n),
        'TECHNICAL EXPERTISE': np.random.randint(1, 10, n),
        'PROJECT DELIVERY': np.random.randint(1, 10, n),
        'POST-SALES SUPPORT': np.random.randint(1, 10, n),
        'NPS RATING': np.random.randint(1, 11, n),
        'REVENUE': np.random.randint(50000, 500000, n),
    })
    df['HARDWARE'] = (df['REVENUE'] * np.random.uniform(0.1, 0.25, n)).astype(int)
    df['SOFTWARE'] = (df['REVENUE'] * np.random.uniform(0.15, 0.35, n)).astype(int)
    df['MANPOWER'] = (df['REVENUE'] * np.random.uniform(0.2, 0.4, n)).astype(int)
    df['COGS'] = df['HARDWARE'] + df['SOFTWARE'] + df['MANPOWER']
    df['GROSS_PROFIT'] = df['REVENUE'] - df['COGS']
    df['GROSS_MARGIN'] = df['GROSS_PROFIT'] / df['REVENUE']
    df['AVG_SERVICE'] = df[['PRESALES AND PARTNERSHIP', 'TECHNICAL EXPERTISE', 'PROJECT DELIVERY', 'POST-SALES SUPPORT']].mean(axis=1)

# Dimension Mapping
segmentation_opts = {
    'Industry Sector': 'SECTOR',
    'Client Type': 'TYPE',
    'Organisation Size': 'STAFF STRENGTH',
    'Country': 'COUNTRY'
}

# ---------------------------------------------------------
# 2. Dash Layout
# ---------------------------------------------------------
app = Dash(__name__)
app.title = 'Client Profitability & Cost Intelligence'

logo_mark = html.Div('$', style={
    'width': '40px', 'height': '40px', 'borderRadius': '8px',
    'backgroundColor': '#2B6CB0', 'color': 'white', 'fontSize': '22px',
    'fontWeight': '700', 'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'
})

app.layout = html.Div([
    # Top Header
    html.Div([
        html.Div([
            logo_mark,
            html.Div([
                html.Div('STRATEGIC FINANCIAL ANALYTICS', style={'fontSize': '11px', 'fontWeight': '800', 'letterSpacing': '1.2px', 'color': '#2B6CB0'}),
                html.Div('Client Segment Profitability & Performance Optimization', style={'fontSize': '18px', 'fontWeight': '700', 'color': '#1A202C'})
            ])
        ], style={'display': 'flex', 'alignItems': 'center', 'gap': '12px'})
    ], style={'padding': '14px 28px', 'backgroundColor': 'white', 'borderBottom': '1px solid #E2E8F0'}),

    # Control Toolbar
    html.Div([
        html.Div([
            html.Label('Primary Segment Dimension', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Dropdown(id='dim-control', options=[{'label': k, 'value': v} for k, v in segmentation_opts.items()], value='SECTOR', clearable=False)
        ]),
        html.Div([
            html.Label('Year', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Dropdown(id='year-control', options=[{'label': 'All Years', 'value': 'All'}] + [{'label': str(y), 'value': y} for y in sorted(df['YEAR'].unique())], value='All', clearable=False)
        ]),
        html.Div([
            html.Label('Country Filter', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Dropdown(id='country-control', options=[{'label': 'All Countries', 'value': 'All'}] + [{'label': c, 'value': c} for c in sorted(df['COUNTRY'].unique())], value='All', clearable=False)
        ]),
        html.Div([
            html.Label('Min NPS Score Filter', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Slider(id='nps-control', min=1, max=10, step=1, value=1, marks={i: str(i) for i in range(1, 11)})
        ])
    ], style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr 1fr 1.5fr', 'gap': '16px', 'padding': '14px 28px', 'backgroundColor': '#F7FAFC', 'borderBottom': '1px solid #E2E8F0'}),

    # Dynamic KPI Summary Section
    html.Div(id='kpi-container', style={'display': 'grid', 'gridTemplateColumns': 'repeat(4, 1fr)', 'gap': '16px', 'padding': '16px 28px 0 28px', 'backgroundColor': '#EDF2F7'}),

    # Main Grid Layout for All 3 Charts
    html.Div([
        # Row 1: Profitability & Cost Structures (sbs)
        html.Div([
            html.Div([dcc.Graph(id='chart-1-profitability')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'}),
            html.Div([dcc.Graph(id='chart-2-costs')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'}),
        ], style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '18px'}),

        # Row 2: Service Quality & NPS Matrix
        html.Div([
            html.Div([dcc.Graph(id='chart-3-satisfaction')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'})
        ])
    ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '18px', 'padding': '16px 28px 24px 28px', 'backgroundColor': '#EDF2F7'}),

    # Global Footer
    html.Footer('Enterprise Operations Analytics • Internal Decision Support Dashboard', style={'padding': '12px 28px', 'fontSize': '11px', 'color': '#718096', 'backgroundColor': 'white', 'borderTop': '1px solid #E2E8F0'})
], style={'fontFamily': 'Segoe UI, Arial, sans-serif', 'backgroundColor': '#EDF2F7', 'minHeight': '100vh'})


# ---------------------------------------------------------
# 3. Unified Callback
# ---------------------------------------------------------
@app.callback(
    [
        Output('kpi-container', 'children'),
        Output('chart-1-profitability', 'figure'),
        Output('chart-2-costs', 'figure'),
        Output('chart-3-satisfaction', 'figure')
    ],
    [
        Input('dim-control', 'value'),
        Input('year-control', 'value'),
        Input('country-control', 'value'),
        Input('nps-control', 'value')
    ]
)
def update_dashboard_view(dimension, year_val, country_val, min_nps):
    # Filters
    d = df.copy()
    if year_val != 'All':
        d = d[d['YEAR'] == year_val]
    if country_val != 'All':
        d = d[d['COUNTRY'] == country_val]
    d = d[d['NPS RATING'] >= min_nps]

    # KPIs
    if not d.empty:
        total_rev = d['REVENUE'].sum()
        total_profit = d['GROSS_PROFIT'].sum()
        avg_margin = d['GROSS_MARGIN'].mean() * 100
        avg_service = d['AVG_SERVICE'].mean()
    else:
        total_rev, total_profit, avg_margin, avg_service = 0, 0, 0, 0

    kpi_cards = [
        html.Div([
            html.Div(title, style={'fontSize': '11px', 'color': '#718096', 'fontWeight': '600'}),
            html.Div(val, style={'fontSize': '20px', 'fontWeight': '700', 'color': '#2B6CB0', 'marginTop': '4px'})
        ], style={'backgroundColor': 'white', 'padding': '14px 18px', 'borderRadius': '8px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'})
        for title, val in [
            ("Total Revenue", f"${total_rev:,.0f}"),
            ("Total Gross Profit", f"${total_profit:,.0f}"),
            ("Avg Gross Margin", f"{avg_margin:.1f}%"),
            ("Avg Service Score", f"{avg_service:.1f} / 10")
        ]
    ]

    if d.empty:
        empty_fig = go.Figure().update_layout(title="No clients match the selected filters.", template="plotly_white")
        return kpi_cards, empty_fig, empty_fig, empty_fig

    # ---------------------------------------------------------
    # CHART 1: Gross Profit & Margin Efficiency
    # ---------------------------------------------------------
    agg_profit = d.groupby(dimension)[['GROSS_PROFIT', 'REVENUE', 'GROSS_MARGIN']].agg({
        'GROSS_PROFIT': 'sum',
        'REVENUE': 'sum',
        'GROSS_MARGIN': 'mean'
    }).reset_index().sort_values('GROSS_PROFIT', ascending=True)

    fig1 = px.bar(
        agg_profit,
        x='GROSS_PROFIT',
        y=dimension,
        color='GROSS_MARGIN',
        color_continuous_scale='Viridis',
        orientation='h',
        template='plotly_white',
        title=f'1. Total Gross Profit & Gross Margin % by {dimension}'
    )
    fig1.update_layout(
        height=420,
        margin=dict(t=50, l=110, r=30, b=40),
        coloraxis_colorbar=dict(title="Margin %", tickformat=".0%")
    )
    fig1.update_xaxes(tickformat='$,.0f', title='Gross Profit ($)')
    fig1.update_yaxes(title='')

    # ---------------------------------------------------------
    # CHART 2: Cost Structure Breakdown
    # ---------------------------------------------------------
    agg_cost = d.groupby(dimension)[['HARDWARE', 'SOFTWARE', 'MANPOWER']].sum().reset_index()
    
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=agg_cost[dimension], y=agg_cost['HARDWARE'], name='Hardware', marker_color='#3182CE'))
    fig2.add_trace(go.Bar(x=agg_cost[dimension], y=agg_cost['SOFTWARE'], name='Software', marker_color='#DD6B20'))
    fig2.add_trace(go.Bar(x=agg_cost[dimension], y=agg_cost['MANPOWER'], name='Manpower', marker_color='#38A169'))
    
    fig2.update_layout(
        title=f'2. Cost Composition Breakdown by {dimension}',
        barmode='stack',
        template='plotly_white',
        height=420,
        margin=dict(t=50, l=60, r=30, b=40),
        xaxis_title='',
        yaxis_title="Cost ($)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig2.update_yaxes(tickformat='$,.0f')

    # ---------------------------------------------------------
    # CHART 3: Service Quality vs. Profitability Matrix
    # ---------------------------------------------------------
    agg_satisfaction = d.groupby(dimension).agg({
        'AVG_SERVICE': 'mean',
        'GROSS_MARGIN': 'mean',
        'REVENUE': 'sum',
        'NPS RATING': 'mean'
    }).reset_index()

    fig3 = go.Figure()
    
    for seg in agg_satisfaction[dimension]:
        sub = agg_satisfaction[agg_satisfaction[dimension] == seg]
        fig3.add_trace(go.Scatter(
            x=sub['AVG_SERVICE'],
            y=sub['GROSS_MARGIN'] * 100,
            mode='markers+text',
            name=str(seg),
            text=sub[dimension],
            textposition="top center",
            marker=dict(
                size=np.clip(np.sqrt(sub['REVENUE'] / 50000) * 8, 14, 45),
                opacity=0.8,
                line=dict(width=1, color='white')
            ),
            customdata=np.stack((sub['REVENUE'], sub['NPS RATING']), axis=-1),
            hovertemplate="<b>%{text}</b><br>" +
                          "Avg Service Score: %{x:.2f} / 10<br>" +
                          "Gross Margin: %{y:.1f}%<br>" +
                          "Total Revenue: $%{customdata[0]:,.0f}<br>" +
                          "Avg NPS Rating: %{customdata[1]:.1f}<extra></extra>"
        ))

    # Median Target Lines
    fig3.add_vline(x=agg_satisfaction['AVG_SERVICE'].median(), line_dash='dash', line_color='#A0AEC0',
                   annotation_text='Median Quality', annotation_position='top right')
    fig3.add_hline(y=agg_satisfaction['GROSS_MARGIN'].median() * 100, line_dash='dash', line_color='#A0AEC0',
                   annotation_text='Median Margin', annotation_position='bottom right')

    fig3.update_layout(
        title=f'3. Service Quality Score vs Gross Margin (%) by {dimension} (Bubble Size = Total Revenue)',
        template='plotly_white',
        height=450,
        margin=dict(t=50, l=60, r=30, b=40),
        xaxis=dict(title='Average Service Score (1–10)'),
        yaxis=dict(title='Gross Margin (%)'),
        showlegend=True
    )

    return kpi_cards, fig1, fig2, fig3


if __name__ == '__main__':
    app.run(debug=True)

# Strategic Analysis

## 1. Which Sectors Are the Strongest Overall Performers?

*Using Chart 1: Profitability & Margin*

### Insight

**Healthcare**, **Info Tech**, and **Manufacturing** are the strongest-performing sectors because they combine:

- High gross profit
- Strong gross margins

This means they are not only generating large amounts of revenue but are also converting revenue into profit efficiently.

### Action

The company should:

- Prioritise acquiring and retaining clients in these sectors.
- Allocate more sales and marketing resources to these industries.
- Develop sector-specific offerings for these clients.

---

## 2. Which Sectors & Segments Have Margin Improvement Opportunities?

*Using Chart 1: Profitability & Margin + Chart 2: Cost Composition*

### Sector-Level Breakdown

- **Finance:**
  - **Margin:** ~43% (Healthy)
  - **Profit:** Relatively low (~$4M)
  - **Takeaway:** High efficiency and strong unit economics, but under-scaled in profit volume.

- **Education:**
  - **Margin:** ~38%
  - **Profit:** Moderate (~$10M)
  - **Takeaway:** Stable volume baseline, but suffers from mild margin compression due to labour and software costs.

- **Charity & NPO:**
  - **Margin:** Very low (~14–15%)
  - **Profit:** Almost negligible (<$2M total profit)
  - **Takeaway:** Low volume combined with severely compressed margins, resulting in a poor return on deployed resources.

### Cross-Segment Opportunities

- **Singapore Operations:** Generates massive gross profit volume (~$173M), but has a compressed gross margin of ~39–40% due to high localised manpower (~$93M) and software overhead (~$81M).
- **Small Business Segment (1–49 Staff):** Yields the lowest margin efficiency (~32–34%) across organisation sizes due to fixed delivery overheads relative to smaller contract values.

### Action Plan

- **Low-Margin Sectors (Charity / NPO):** Re-evaluate strategic alignment. If retained, transition these accounts to standardised, self-service packages with minimal custom labour hours to safeguard profitability.
- **Singapore Margin Expansion:** Focus on vendor cost renegotiations, software licensing, and manpower deployment efficiencies specifically in Singapore. A 2–3% margin improvement could generate millions in additional gross profit.
- **SMB Operational Shift (1–49 Staff):** Automate client onboarding and service delivery to lift SMB margins closer to the enterprise average (~42–45%).

---

## 3. What Drives Operational Costs?

*Using Chart 2: Cost Structure Breakdown*

Across virtually all client categories, including sectors, countries, and organisation sizes, operating expenses follow a consistent hierarchy:

1. **Manpower (Green):** Consistently the **largest single cost driver** across every major high-volume category.
2. **Software (Red/Orange):** Represents the second-largest cost driver and scales heavily with enterprise accounts.
3. **Hardware (Blue):** Remains the smallest contributor to overall COGS.

### Category Highlights

- **Healthcare & Info Tech:** Heavily burdened by high labour expenses (~$28M–$30M each) and significant software licensing costs (~$21M–$25M each).
- **Private Sector & Enterprise (>200 Staff):** Manpower costs significantly exceed other expenses, reaching more than **$100M** in Private accounts and approximately **$67M** in Enterprise accounts.

### Key Insight

Gross profit margin performance is strongly influenced by **manpower efficiency and resource utilisation**. High total cost is not necessarily negative when paired with sufficient scale, but unoptimised labour allocation can rapidly erode margins in smaller or service-heavy accounts.

### Action Plan

- **Workforce Planning & Automation:** Standardise implementation workflows and introduce automated service tools in high-labour sectors such as Healthcare and IT.
- **Software License Optimisation:** Negotiate bulk, centralised software vendor contracts to reduce the secondary cost driver (~$85M spent in the Private sector alone).
- **Delivery Rebalancing:** Utilise offshore support teams or mixed delivery models for enterprise clients (>200 staff) to reduce average delivery costs.

> **Note:** Because labour is the dominant cost, even a **3–5% reduction in manpower delivery hours** could contribute directly to margin expansion.

---

## 4. Does Service Quality Affect Profitability?

*Using Chart 3: Service Quality vs. Profitability Matrix*

### High-Performing Quadrants

| Sector / Segment | Avg Service Quality | Gross Margin (%) | Revenue Scale |
|---|---:|---:|---|
| **Info Tech** | High (~8.2+) | **Highest (~45%+)** | Large |
| **Transportation** | High (~8.0+) | **Highest (~45%+)** | Moderate |
| **Manufacturing** | Strong (~7.8+) | **Highest (~45%+)** | Large |
| **Finance** | **Highest (~8.5+)** | Healthy (~43%) | Small |

### Outlier Quadrants

| Sector / Segment | Avg Service Quality | Gross Margin (%) | Revenue Scale |
|---|---:|---:|---|
| **Charity / NPO** | High (~8.0+) | **Low (~14–15%)** | Very Small |

### Key Insight

There is a clear positive relationship between service quality scores (`AVG_SERVICE` / `NPS`) and gross profit margins:

- Top-performing sectors such as **Info Tech, Transportation, and Manufacturing** achieve higher gross margins while maintaining high customer satisfaction scores.

Investing in client experience, particularly in **Technical Expertise**, **Project Delivery**, and **Post-Sales Support**, can reinforce premium pricing power, client retention, and margin stability.

> **Note:** The **Charity** sector demonstrates that high customer satisfaction alone cannot overcome poor pricing power or an inflated cost-to-serve ratio.

---

## Summary & Conclusion

To answer the main problem statement, **Healthcare**, **Info Tech**, **Manufacturing**, and **Large Enterprise Clients (>200 Staff)** represent the company's most valuable segments. They drive the majority of gross profit volume, sustain strong gross margins (~42–45%), and maintain above-average service satisfaction ratings.

### Strategic Action Matrix

| Priority Pillar | High-Value Focus Segments | Key Strategic Directive |
|---|---|---|
| **Acquisition & Scale** | Info Tech, Healthcare, Govt | Expand footprint and prioritise retention |
| **High-Margin Niche** | China (~52% GM), Finance (~43% GM) | Scale sales volume to capitalise on high GM% |
| **Cost Optimisation** | Singapore & Private Accounts | Optimise manpower hours and software COGS |
| **Margin Restructuring** | SMBs (1–49 Staff) & NPOs/Charities | Standardise workflows and simplify offers |

### Final Takeaways

- **Prioritise High-Value Segments:** Concentrate client acquisition, dedicated account managers, and retention programmes on **Healthcare, IT, Manufacturing, and Enterprise accounts**.
- **Drive Labour Efficiency:** Focus operational improvement on **manpower utilisation**, particularly in Singapore and large accounts where labour expenses are highest.
- **Maintain Service Excellence:** High service quality supports healthy margin realisation. Maintaining quality standards across presales, delivery, and post-sales support can safeguard recurring revenue and long-term business growth.